<a href="https://colab.research.google.com/github/YuZh98/finetune-gsm8k/blob/main/notebooks/02_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluate: GSM8K test set

Colab front-end for `src/eval_gsm8k.py`. Greedy decoding, exact-match on the final numeric answer.

**First run the base model**, then each adapter.

## 1. Mount Drive + Clone + Install

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_RUNS = '/content/drive/MyDrive/finetune-gsm8k-runs'
DRIVE_RESULTS = '/content/drive/MyDrive/finetune-gsm8k-runs/results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

if not os.path.isdir('/content/finetune-gsm8k'):
    !git clone https://github.com/YuZh98/finetune-gsm8k.git /content/finetune-gsm8k
os.chdir('/content/finetune-gsm8k')
!pip install -q -r requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Smoke test: 16 problems against the base model

Verify the harness works before running the full 1319 problems.

In [8]:
from src.eval_gsm8k import evaluate

metrics = evaluate(adapter_path=None, limit=16, batch_size=4)
metrics

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

GSM8K eval: 100%|██████████| 4/4 [03:36<00:00, 54.17s/it]


{'n_problems': 16, 'n_correct': 11, 'accuracy': 0.6875, 'elapsed_sec': 216.7}

## 3. Full eval: base model

In [9]:
from src.eval_gsm8k import evaluate, append_row
from src.config import ANSWER_EXTRACTOR_VERSION

CSV_PATH = f'{DRIVE_RESULTS}/runs.csv'

metrics = evaluate(adapter_path=None, limit=None, batch_size=8)
append_row(CSV_PATH, {
    'run_id': 'run0_base',
    'adapter': '',
    'extractor': ANSWER_EXTRACTOR_VERSION,
    **metrics,
})
print(f'✓ Base: {metrics["accuracy"]:.4f} ({metrics["n_correct"]}/{metrics["n_problems"]})')

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

GSM8K eval: 100%|██████████| 165/165 [2:19:44<00:00, 50.81s/it]


✓ Base: 0.8120 (1071/1319)


## 4. Full eval: trained adapters

Loops over all adapters found on Drive. Or set `RUN_IDS` manually to eval a subset.

In [10]:
from src.eval_gsm8k import evaluate
  # Pick smallest adapter, limit to 32 problems
smoke = evaluate(
      adapter_path=f'{DRIVE_RUNS}/run1_r8/adapter',
      limit=32,
      batch_size=8,
)
print(f'run1_r8 smoke: {smoke["accuracy"]:.3f} ({smoke["n_correct"]}/{smoke["n_problems"]})')

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

GSM8K eval: 100%|██████████| 4/4 [02:46<00:00, 41.65s/it]


run1_r8 smoke: 0.625 (20/32)


Expected: ≥ 0.6 (base on first 32 likely also high).

Red flags (if smoke fails):
  - < 0.3 → adapter emits unrecognized format. Dump completions before full run.
  - Exactly 0.0 → adapter loading broken, not extractor.
  - 0.5-0.7 → ambiguous; run a 100-problem smoke or dump 5 completions to check.

In [11]:
# Dump-completions cell (only run if smoke looks off):

from src.utils import load_tokenizer, load_model_for_eval
from src.eval_gsm8k import build_prompts, generate_batch
from src.config import EVAL_MAX_NEW_TOKENS
from src.answer_parsing import extract_answer
from datasets import load_dataset

tok = load_tokenizer(padding_side="left")
model = load_model_for_eval(f'{DRIVE_RUNS}/run1_r8/adapter')
test = load_dataset("openai/gsm8k", "main", split="test").select(range(5))
prompts = build_prompts(tok, [ex["question"] for ex in test])
comps = generate_batch(model, tok, prompts, EVAL_MAX_NEW_TOKENS)
for i, c in enumerate(comps):
    print(f"--- {i} (extracted={extract_answer(c)}) ---")
    print(c[-300:])

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

--- 0 (extracted=18) ---
every morning.
3. This leaves 16 - 3 = 13 eggs remaining.
4. She bakes muffins with 4 eggs each day.
5. This means she has 13 - 4 = 9 eggs left to sell.
6. She sells the remaining eggs at $2 per egg.
7. Therefore, she makes 9 * $2 = $18 every day at the farmers' market.

Therefore, the answer is 18.
--- 1 (extracted=3) ---
f blue fiber.
2. It also requires half as much white fiber as blue fiber, which means it needs 2 / 2 = 1 bolt of white fiber.
3. To find the total number of bolts needed, we add the blue and white fiber together: 2 + 1 = 3 bolts.

Therefore, it takes 3 bolts in total to make the robe.

#### N
#### N
--- 2 (extracted=70000) ---
 find the final value of the house by adding the increase in value to the initial value: $120,000 + $80,000 = $200,000.
4. Finally, we can calculate the profit by subtracting the total cost from the final value: $200,000 - $130,000 = $70,000.

Therefore, Josh made a profit of $70,000.

#### N
#### N
--- 3 (extracted=540

In [12]:
from src.eval_gsm8k import evaluate, append_row
from src.config import ANSWER_EXTRACTOR_VERSION

CSV_PATH = f'{DRIVE_RESULTS}/runs.csv'

# Auto-detect adapters on Drive, or set manually:
# RUN_IDS = ['run1_r8', 'run2_r16', 'run3_r64', 'run4_mlp', 'run5_lr_low', 'run6_lr_high', 'run7_data5k']
RUN_IDS = sorted([
    d for d in os.listdir(DRIVE_RUNS)
    if os.path.isdir(f'{DRIVE_RUNS}/{d}/adapter')
])
print(f'Found adapters: {RUN_IDS}')

for run_id in RUN_IDS:
    adapter = f'{DRIVE_RUNS}/{run_id}/adapter'
    print(f'\nEvaluating {run_id}...')
    metrics = evaluate(adapter_path=adapter, limit=None, batch_size=8)
    append_row(CSV_PATH, {
        'run_id': run_id,
        'adapter': adapter,
        'extractor': ANSWER_EXTRACTOR_VERSION,
        **metrics,
    })
    print(f'  ✓ {run_id}: {metrics["accuracy"]:.4f} ({metrics["n_correct"]}/{metrics["n_problems"]})')

Found adapters: ['run1_r8', 'run2_r16', 'run3_r64', 'run4_mlp', 'run5_lr_low', 'run6_lr_high', 'run7_data5k']

Evaluating run1_r8...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

GSM8K eval: 100%|██████████| 165/165 [1:58:22<00:00, 43.05s/it]


  ✓ run1_r8: 0.7286 (961/1319)

Evaluating run2_r16...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

GSM8K eval: 100%|██████████| 165/165 [1:54:18<00:00, 41.57s/it]


  ✓ run6_lr_high: 0.7104 (937/1319)

Evaluating run7_data5k...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

GSM8K eval: 100%|██████████| 165/165 [2:01:40<00:00, 44.24s/it]


  ✓ run7_data5k: 0.7210 (951/1319)


In [ ]:
from src.eval_gsm8k import evaluate, append_row
from src.config import ANSWER_EXTRACTOR_VERSION

adapter = f'{DRIVE_RUNS}/run2_r16_seed43/adapter'
metrics = evaluate(adapter_path=adapter, limit=None, batch_size=8)
append_row(f'{DRIVE_RESULTS}/runs.csv', {
    'run_id': 'run2_r16_seed43',
    'adapter': adapter,
    'extractor': ANSWER_EXTRACTOR_VERSION,
    **metrics,
})
print(f'✓ run2_r16_seed43: {metrics["accuracy"]:.4f}')

## 5. Inspect results

In [13]:
import pandas as pd

df = pd.read_csv(f'{DRIVE_RESULTS}/runs.csv')
df.sort_values('accuracy', ascending=False)

,run_id,adapter,answer_regex,n_problems,n_correct,accuracy,elapsed_sec
5,run0_base,NaN,multi-pattern-v1,1319,1071,0.811979,8384.2
6,run1_r8,/content/drive/MyDrive/finetune-gsm8k-runs/run...,multi-pattern-v1,1319,961,0.728582,7102.9
12,run7_data5k,/content/drive/MyDrive/finetune-gsm8k-runs/run...,multi-pattern-v1,1319,951,0.721001,7300.4
8,run3_r64,/content/drive/MyDrive/finetune-gsm8k-runs/run...,multi-pattern-v1,1319,943,0.714936,7494.8
11,run6_lr_high,/content/drive/MyDrive/finetune-gsm8k-runs/run...,multi-pattern-v1,1319,937,0.710387,6858.2
10,run5_lr_low,/content/drive/MyDrive/finetune-gsm8k-runs/run...,multi-pattern-v1,1319,932,0.706596,7819.8
7,run2_r16,/content/drive/MyDrive/finetune-gsm8k-runs/run...,multi-pattern-v1,1319,829,0.628506,8198.9
9,run4_mlp,/content/drive/MyDrive/finetune-gsm8k-runs/run...,multi-pattern-v1,1319,808,0.612585,8757.9
1,run1_r8,/content/drive/MyDrive/finetune-gsm8k-runs/run...,####\s*(-?\d+(?:\.\d+)?),1319,549,0.416224,7119.6
3,run1_r8,/content/drive/MyDrive/finetune-gsm8k-runs/run...,####\s*(-?\d+(?:\.\d+)?),1319,549,0.416224,7123.5
